In [202]:
# hide-output

# show → code input visible by default
# hide-output → output hidden by default
# show hide-output → both (can combine on one line)

# Imports: data handling, visualization, and BigQuery connector
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
#from google.cloud import bigquery
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html

In [203]:
# hide-output
# Configure query parameters and estimate BigQuery cost for the seed evaluation query
query_location = './sql/abtestseed.sql'
parameters = {
    'lookback_days': 28,  # number of days to look back for user activity after the AB test start date
}

bqc = BigQueryConnector()
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 623.1 MB when run.
Estimated query cost: $0.00


In [204]:
# hide-output
# Fetch seed evaluation data from BigQuery or load from local pickle cache
import os

force_refresh = False  # Set to True to force re-querying the data and updating the pickle file

if not os.path.exists('./data/abtestseed.pkl') or force_refresh:
    data = bqc.get(query=query_location, is_path=True, query_parameters=parameters)
    data.to_pickle('./data/abtestseed.pkl')
else:
    data = pd.read_pickle('./data/abtestseed.pkl')
    print("Pickle already exists, skipping query.")

Pickle already exists, skipping query.


In [205]:
# Preview raw seed evaluation data
data

,salt,bucket,DAU,DPU_0,DPU_10,iapnetrevenue,adrevenue,ARPDAU,AD_ARPDAU,ARPDAU_DIFF,AD_ARPDAU_DIFF,ABS_ARPDAU_DIFF,ABS_AD_ARPDAU_DIFF,abTestWeightedBucketsBase64,dt_start,dt_end
0,004a46868,0,49975.241379,1808.586207,307.068966,347651.043589,127249.248281,0.239454,0.087828,0.000000,0.000000,0.000000,0.000000,W1t7Im1pbiI6MCwibWF4Ijo0OX1dLFt7Im1pbiI6NTAsIm...,2026-02-26,2026-03-26
1,004a46868,1,49857.931034,1814.586207,308.724138,352445.977820,127584.640518,0.243288,0.088278,0.016012,0.005120,0.016012,0.005120,W1t7Im1pbiI6MCwibWF4Ijo0OX1dLFt7Im1pbiI6NTAsIm...,2026-02-26,2026-03-26
2,026e63450,0,49948.827586,1824.000000,311.137931,351135.286245,127583.990802,0.241924,0.088113,0.000000,0.000000,0.000000,0.000000,W1t7Im1pbiI6MCwibWF4Ijo0OX1dLFt7Im1pbiI6NTAsIm...,2026-02-26,2026-03-26
3,026e63450,1,49884.344828,1799.172414,304.655172,348961.735165,127249.897997,0.240817,0.087992,-0.004575,-0.001375,0.004575,0.001375,W1t7Im1pbiI6MCwibWF4Ijo0OX1dLFt7Im1pbiI6NTAsIm...,2026-02-26,2026-03-26
4,02d73275d,0,49961.344828,1793.724138,298.344828,341375.352703,126927.966962,0.235125,0.087642,0.000000,0.000000,0.000000,0.000000,W1t7Im1pbiI6MCwibWF4Ijo0OX1dLFt7Im1pbiI6NTAsIm...,2026-02-26,2026-03-26
5,02d73275d,1,49871.827586,1829.448276,317.448276,358721.668707,127905.921837,0.247625,0.088464,0.053160,0.009381,0.053160,0.009381,W1t7Im1pbiI6MCwibWF4Ijo0OX1dLFt7Im1pbiI6NTAsIm...,2026-02-26,2026-03-26
6,02e259f93,0,49607.586207,1776.379310,299.517241,345573.761223,125870.304322,0.239766,0.087524,0.000000,0.000000,0.000000,0.000000,W1t7Im1pbiI6MCwibWF4Ijo0OX1dLFt7Im1pbiI6NTAsIm...,2026-02-26,2026-03-26
7,02e259f93,1,50225.586207,1846.793103,316.275862,354523.260187,128963.584477,0.242955,0.088575,0.013298,0.012005,0.013298,0.012005,W1t7Im1pbiI6MCwibWF4Ijo0OX1dLFt7Im1pbiI6NTAsIm...,2026-02-26,2026-03-26
8,0396b92a2,0,49946.620690,1812.379310,302.206897,346378.138927,126983.836978,0.238716,0.087705,0.000000,0.000000,0.000000,0.000000,W1t7Im1pbiI6MCwibWF4Ijo0OX1dLFt7Im1pbiI6NTAsIm...,2026-02-26,2026-03-26
9,0396b92a2,1,49886.551724,1810.793103,313.586207,353718.882483,127850.051821,0.244032,0.088401,0.022271,0.007938,0.022271,0.007938,W1t7Im1pbiI6MCwibWF4Ijo0OX1dLFt7Im1pbiI6NTAsIm...,2026-02-26,2026-03-26


In [206]:
# Select the salt with the minimum ARPDAU difference between buckets (best balanced seed)
minarpdaudiff = data[data['bucket'] == 1]['ABS_ARPDAU_DIFF'].min()
selected_salt = data[data['ABS_ARPDAU_DIFF'] == minarpdaudiff]['salt']
selected_data = data[data['salt'] == selected_salt.iloc[0]]

selected_data

,salt,bucket,DAU,DPU_0,DPU_10,iapnetrevenue,adrevenue,ARPDAU,AD_ARPDAU,ARPDAU_DIFF,AD_ARPDAU_DIFF,ABS_ARPDAU_DIFF,ABS_AD_ARPDAU_DIFF,abTestWeightedBucketsBase64,dt_start,dt_end
12,06a5ad920,0,49757.068966,1802.000000,302.758621,348517.903989,125720.749340,0.241155,0.087153,0.00000,0.000000,0.00000,0.000000,W1t7Im1pbiI6MCwibWF4Ijo0OX1dLFt7Im1pbiI6NTAsIm...,2026-02-26,2026-03-26
13,06a5ad920,1,50076.103448,1821.172414,313.034483,351579.117420,129113.139459,0.241584,0.088947,0.00178,0.020581,0.00178,0.020581,W1t7Im1pbiI6MCwibWF4Ijo0OX1dLFt7Im1pbiI6NTAsIm...,2026-02-26,2026-03-26


In [207]:
# hide-output
# Configure metrics query using the selected seed and estimate BigQuery cost
query_location = './sql/abtestseedmetrics.sql'
parameters = {
    'lookback_days': 28,  # needs to reach Feb 8 2025 for the "same time last year" baseline
    #'seed': '08dc1a6de'
    'seed': selected_salt.iloc[0]  # IGNORE - old seed value that was replaced in the SQL file
}

bqc = BigQueryConnector()
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 623.1 MB when run.
Estimated query cost: $0.00


In [208]:
# Fetch daily metrics data from BigQuery or load from local pickle cache
force_refresh = True  # Set to True to force re-querying the data and updating the pickle file

if not os.path.exists('./data/abtestseedmetrics.pkl') or force_refresh:
    datametrics = bqc.get(query=query_location, is_path=True, query_parameters=parameters)
    datametrics.to_pickle('./data/abtestseedmetrics.pkl')
else:
    datametrics = pd.read_pickle('./data/abtestseedmetrics.pkl')
    print("Pickle already exists, skipping query.")

In [209]:
# hide-output
# Preview daily metrics by variant
datametrics

,dt,variant,users,iapnetrevenue,iapnetrevenue_diff,adrevenue,adrevenue_diff,arpdau,arpdau_diff,adrpdau,adarpdau_diff
0,2026-02-26,0,52407,11145.394602,-0.018309,4464.550176,-0.021457,0.212670,-0.018309,0.085190,-0.021457
1,2026-02-26,1,52671,11349.461112,NaN,4560.345439,NaN,0.215478,NaN,0.086582,NaN
2,2026-02-27,0,51790,19997.187681,-0.040089,4213.479537,-0.030714,0.386121,-0.040089,0.081357,-0.030714
3,2026-02-27,1,51811,20798.862042,NaN,4342.892329,NaN,0.401437,NaN,0.083822,NaN
4,2026-02-28,0,51040,13064.677940,-0.008431,4427.438986,-0.032046,0.255969,-0.008431,0.086744,-0.032046
5,2026-02-28,1,51639,13174.825760,NaN,4569.318902,NaN,0.255133,NaN,0.088486,NaN
6,2026-03-01,0,51828,13072.721237,0.005528,4693.521833,-0.024052,0.252233,0.005528,0.090560,-0.024052
7,2026-03-01,1,52210,13000.454422,NaN,4806.410296,NaN,0.249003,NaN,0.092059,NaN
8,2026-03-02,0,51663,9447.857886,0.005719,4181.958925,-0.038306,0.182875,0.005719,0.080947,-0.038306
9,2026-03-02,1,52228,9393.828011,NaN,4342.151299,NaN,0.179862,NaN,0.083138,NaN


In [210]:
# Plot ARPDAU by variant over time
fig = px.line(datametrics, x='dt', y='arpdau', color='variant',
              title='ARPDAU Diff by Variant Over Time',
              labels={'dt': 'Date', 'arpdau_diff': 'ARPDAU Diff', 'variant': 'Variant'},
              markers=True)

            # Update layout for better readability
fig.update_layout(height=500, width=1000)

fig.show()

In [211]:
# Plot daily ARPDAU difference vs. control with threshold reference lines
fig = px.line(datametrics, x='dt', y='arpdau_diff', color='variant',
              title='ARPDAU Diff by Variant Over Time',
              labels={'dt': 'Date', 'arpdau_diff': 'ARPDAU Diff', 'variant': 'Variant'},
              markers=True)
            # Add a zero line as reference
fig.add_hline(y=0, line_dash="dash", line_color="gray", annotation_text="0")
fig.add_hline(y=0.02, line_dash="dash", line_color="orange", annotation_text="0.2")
fig.add_hline(y=-0.02, line_dash="dash", line_color="orange", annotation_text="-0.2")
fig.add_hline(y=0.05, line_dash="dash", line_color="red", annotation_text="0.5")
fig.add_hline(y=-0.05, line_dash="dash", line_color="red", annotation_text="-0.5")

            # Update layout for better readability
fig.update_layout(height=500, width=1000)

fig.show()

In [212]:
# Violin plot of ARPDAU distribution per variant with mean and median annotations
# Create a distribution plot with mean and median marks
fig = go.Figure()

# Add violin plots for each variant
for variant in sorted(datametrics['variant'].unique()):
    variant_data = datametrics[datametrics['variant'] == variant]['arpdau']
    fig.add_trace(go.Violin(
        y=variant_data,
        name=f'Variant {variant}',
        box_visible=True,
        meanline_visible=True,
        points='outliers'  # Show outlier points
    ))

# Calculate means and medians for annotations
variant_stats = datametrics.groupby('variant')['arpdau'].agg(['mean', 'median']).reset_index()
mean_diff = variant_stats.loc[1, 'mean'] - variant_stats.loc[0, 'mean']
relative_diff = (mean_diff / variant_stats.loc[0, 'mean']) * 100

# Add annotations for mean and median values
for idx, row in variant_stats.iterrows():
    fig.add_annotation(
        x=idx,
        y=row['mean'],
        text=f"μ: {row['mean']:.4f}",
        showarrow=False,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=2,
        arrowcolor="green"
    )
    fig.add_annotation(
        x=idx,
        y=row['median'],
        text=f"Median: {row['median']:.4f}",
        showarrow=False,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=2,
        arrowcolor="blue"
    )

fig.update_layout(
    title=f'ARPDAU Distribution by Variant<br><sub>Mean Difference: {mean_diff:.4f} ({relative_diff:.2f}%)</sub>',
    yaxis_title='ARPDAU',
    xaxis_title='Variant',
    violinmode='overlay',
    height=600,
    width=900,
    showlegend=True
)

fig.show()

In [213]:
# Plot daily ad ARPDAU difference vs. control with threshold reference lines
fig = px.line(datametrics, x='dt', y='adarpdau_diff', color='variant',
              title='ADRPDAU Diff by Variant Over Time',
              labels={'dt': 'Date', 'adarpdau_diff': 'ADRPDAU Diff', 'variant': 'Variant'},
              markers=True)
            # Add a zero line as reference
fig.add_hline(y=0, line_dash="dash", line_color="gray", annotation_text="0")
fig.add_hline(y=0.02, line_dash="dash", line_color="orange", annotation_text="0.2")
fig.add_hline(y=-0.02, line_dash="dash", line_color="orange", annotation_text="-0.2")
fig.add_hline(y=0.05, line_dash="dash", line_color="red", annotation_text="0.5")
fig.add_hline(y=-0.05, line_dash="dash", line_color="red", annotation_text="-0.5")

            # Update layout for better readability
fig.update_layout(height=500, width=1000)

fig.show()

In [214]:
# Aggregate metrics by variant and compute absolute and relative differences across the full period
datametricstotal = datametrics.groupby('variant').agg(
    arpdau=('arpdau', 'mean'),
    iapnetrevenue=('iapnetrevenue', 'sum'),
    adrevenue=('adrevenue', 'sum')  
    ).reset_index()

# Calculate relative differences for all metrics in datametricstotal
variant_0 = datametricstotal[datametricstotal['variant'] == 0].iloc[0]
variant_1 = datametricstotal[datametricstotal['variant'] == 1].iloc[0]

metrics_comparison = pd.DataFrame({
    'Metric': ['arpdau', 'iapnetrevenue', 'adrevenue'],
    'Variant 0': [variant_0['arpdau'], variant_0['iapnetrevenue'], variant_0['adrevenue']],
    'Variant 1': [variant_1['arpdau'], variant_1['iapnetrevenue'], variant_1['adrevenue']],
})

# Calculate absolute and relative differences
metrics_comparison['Absolute Diff'] = metrics_comparison['Variant 1'] - metrics_comparison['Variant 0']
metrics_comparison['Relative Diff %'] = (metrics_comparison['Absolute Diff'] / metrics_comparison['Variant 0'])

metrics_comparison

,Metric,Variant 0,Variant 1,Absolute Diff,Relative Diff %
0,arpdau,0.241155,0.241584,0.000429,0.001780
1,iapnetrevenue,348517.903989,351579.117420,3061.213431,0.008784
2,adrevenue,125720.749340,129113.139459,3392.390119,0.026984


In [215]:
export_notebook_html(
    notebook_path='./abtestseed.ipynb',
    output_path='./notebook_output.html',
)

Saved to notebook_output.html


PosixPath('notebook_output.html')